#  atoMLtype ModelOutput Analysis

In this tutorial, we’ll dive deep into the analysis tools provided by the `PredictionRecord` interface of the ModelOutput in this project. This includes:

- Error diagnosis (accuracy, mismatches, confidence)
- Visual analysis (heatmaps, confusion matrices, model embeddings)
- Atom-specific inspection and molecule rendering

These tools are designed to help you understand where your model succeeds, where it fails, and why.

## Load molecules, train model and perform inference.

Before introducing the analysis tools, we will repeat the process of building our model and performing inference from the "Tutorial_quickstart.ipynb"

In [ ]:
from torch.utils.data import random_split
import numpy as np
import logging

logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("dataset_loader.log"),
        logging.StreamHandler()
    ]
)


from atoMLtype.datasets.GNNdataset import GNNdataset
from atoMLtype.models.ModelEncoder import ModelEncoder
from atoMLtype.analysis.accuracy_counts import plot_atom_distribution
from atoMLtype.models.GNN.DMPNNmodel import AtomBondMPNN
from atoMLtype.models.ModelTrainer import GNNTrainer
from atoMLtype.models.ModelEngine import ModelEngine


In [ ]:
dataset_encoder = ModelEncoder(collapse=True)

In [ ]:
# Load the ZINC files
zinc_sdf_path = "./data/parm_at_Frosst/zinc.sdf"
zinc_json_labels = "./data/antechamber/atomLabels_gaff2.json"

zinc_dataset = GNNdataset(sdf_path=zinc_sdf_path, 
                          label_path=zinc_json_labels, 
                          directed_graph=True,
                          labeled=True,
                          encoder=dataset_encoder)

In [ ]:
# Split Train and test dataset (90% train, 10% test)
train_size = int(0.90 * len(zinc_dataset))
test_size = len(zinc_dataset) - train_size
train_dataset, test_dataset = random_split(zinc_dataset, [train_size, test_size])

In [ ]:

atom_node_dim = train_dataset[0].x.shape[1]
bond_edge_dim = train_dataset[0].edge_attr.shape[1]

AtomMPNN_zinc = AtomBondMPNN(atom_input_dim=atom_node_dim, 
                             bond_input_dim=bond_edge_dim, 
                             hidden_dim=512,
                             encoder=dataset_encoder, 
                             num_layers=10,
                             use_attention=True)

In [ ]:
trainer = GNNTrainer(AtomMPNN_zinc, 
                     dataset=train_dataset, 
                     batch_size=32, learning_rate=0.001,
                     epochs=5, 
                     k_folds=5, 
                     random_seed=21)

In [ ]:
training_loss_ouput = trainer.train(draw_curve=False,
                                    verbose=False,
                                    report_step=5
                                    )

In [ ]:
modelEngine_zinc = ModelEngine(model=AtomMPNN_zinc, 
                          dataset=test_dataset, 
                          device="cpu",
                          batch_size=32)

In [ ]:
pred_record = modelEngine_zinc.predict(analysis=True)

pred_record.summary()

## Part 1. Accuracy and Class Distribution

**Functions from:** `accuracy_counts.py`

### Goals:
- Calculate atom-type and element-level accuracy
- Visualize atom-type distributions

### Functions:
- `get_accuracies_and_counts(pred_record)`
- `plot_atom_distribution(pred_record.true_labels)`

In [ ]:
from atoMLtype.analysis.accuracy_counts import get_accuracies_and_counts, plot_atom_distribution

# Display distribution of true labels
plot_atom_distribution(pred_record.true_labels)

# Compute and print accuracy metrics
metrics = get_accuracies_and_counts(pred_record)
for k, v in metrics.items():
    print(f"\n=== {k.upper()} ===")
    print(v if not isinstance(v, dict) else dict(sorted(v.items())))

## Part 2. Confusion Matrices for Misclassifications

**Functions from:** `confusionMatrices.py`

### Goals:
- Generate heatmaps for **intra-** and **cross-element** misclassifications
- See how model confuses similar atom types within a chemical context

### Functions:
- `plot_full_confusion_matrices(pred_record)`
- `plot_element_confusion_matrices(pred_record)`

In [ ]:
from atoMLtype.analysis.confusionMatrices import plot_full_confusion_matrices, plot_element_confusion_matrices

plot_full_confusion_matrices(pred_record)
plot_element_confusion_matrices(pred_record)

## Part 3. Normalized Heatmaps of All Predictions

**Functions from:** `heatmaps.py`

### Goals:
- Show full row-normalized confusion matrix (%-based)
- Highlight systematic bias in prediction classes

### Functions:
- `plot_full_heatmap(pred_record)`
- `plot_element_heatmap(pred_record)`
- `plot_cross_element_heatmap(pred_record)`

In [ ]:
from atoMLtype.analysis.heatmaps import plot_full_heatmap, plot_element_heatmap, plot_cross_element_heatmap

plot_full_heatmap(pred_record)
plot_element_heatmap(pred_record)
plot_cross_element_heatmap(pred_record)

## Part 4. Discrepancy Diagnostics

**Functions from:** `discrepancies.py`

### Goals:
- Quantify errors by molecule and atom type
- Compare element-level performance
- Visualize discrepancy rates and confidence variation

### Functions:
- `plot_confidence_by_pred_label(pred_record)`
- `plot_discrepancy_distribution(pred_record)`
- `plot_element_discrepancy_rate(pred_record)`
- `plot_discrepancy_rate_by_atom_type(pred_record)`

In [ ]:
from atoMLtype.analysis.discrepancies import plot_confidence_by_pred_label, plot_discrepancy_distribution, \
                                             plot_element_discrepancy_rate, plot_discrepancy_rate_by_atom_type

plot_confidence_by_pred_label(pred_record, showfliers=True)
plot_discrepancy_distribution(pred_record)
plot_element_discrepancy_rate(pred_record, valid_elements={"c", "h", "o", "n", "s", "p", "cl", "br", "i", "f"})
plot_discrepancy_rate_by_atom_type(pred_record)

## Part 5. Visualizing Embeddings and Highlighting Errors

**Functions from:** `molecule_embeddings.py`

### Goals:
- Project high-dimensional atom embeddings into 2D space (UMAP or TSNE)
- Overlay mismatched atoms
- Render molecule images with annotated atom-type mismatches

### Functions:
- `visualize_prediction_embeddings(pred_record, method='umap', color_by='true_label')`
- `draw_molecule_with_mismatches_labeled(mol, atom_predictions)`

In [ ]:
from atoMLtype.analysis.molecule_embeddings import visualize_prediction_embeddings, draw_molecule_with_mismatches_labeled

visualize_prediction_embeddings(pred_record, method="tsne", color_by="true_label")

for mol_name, atom_preds in pred_record.mismatched_molecules.items():
    for atom_pred in atom_preds:
        if not atom_pred.pred_label == atom_pred.true_label:
            print(f"{mol_name} atom {atom_pred.atom_idx_in_mol}, pred AT {atom_pred.pred_label} for {atom_pred.true_label}")
    mol = zinc_dataset.get_mol(mol_name)

    img = draw_molecule_with_mismatches_labeled(mol, atom_preds)
    display(img)